In [2]:
import yaml
import os
import polars as pl
from datetime import datetime 
from functools import partial
with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

data_dir = config["data_dir"]
print(data_dir  )
if os.path.exists(data_dir) and os.path.isdir(data_dir):
    list_of_files = os.listdir(data_dir)
else:
    list_of_files = []
print(list_of_files)

E:\smallcaps_data
['AIXI.parquet', 'APUS.parquet', 'BAOS.parquet', 'BEAT.parquet', 'BIAF.parquet', 'CAN.parquet', 'CDTG.parquet', 'CHAI.parquet', 'CHPT.parquet', 'CISS.parquet', 'COOT.parquet', 'CYAB.parquet', 'FBGL.parquet', 'FLYE.parquet', 'GCDT.parquet', 'GIPR.parquet', 'GRI.parquet', 'GRML.parquet', 'IMCC.parquet', 'IMRN.parquet', 'LIDR.parquet', 'logs', 'MEDS.parquet', 'NCPL.parquet', 'NCRA.parquet', 'NEOV.parquet', 'OFAL.parquet', 'OLB.parquet', 'OLOX.parquet', 'output', 'PDSB.parquet', 'PFSA.parquet', 'SBFM.parquet', 'SDEV.parquet', 'SKYQ.parquet', 'SOAR.parquet', 'SPHL.parquet', 'STAK.parquet', 'TNMG.parquet', 'TOPP.parquet', 'TRUG.parquet', 'USDE.parquet', 'VEEA.parquet', 'VRA.parquet', 'VRME.parquet', 'XHLD.parquet', 'XPON.parquet', 'XTNT.parquet', '_catalog.parquet']


In [28]:
list_of_files = [f for f in list_of_files if f.endswith(".parquet")]
from datetime import date

tickers_days = {
    "BEAT": ["2026-10-03","2026-10-04","2026-10-05"],
    "VEEA": ["2026-10-09", "2026-10-05"],
}

columns = ["datetime", "ticker", "open", "high", "low", "close", "volume"]
frames = []

for ticker, days in tickers_days.items():
    parquet_path = os.path.join(data_dir, f"{ticker}.parquet")

    if not os.path.isfile(parquet_path):
        raise FileNotFoundError(f"No existe el paquete Parquet para {ticker}: {parquet_path}")

    requested_days = [date.fromisoformat(day) for day in days]

    ticker_df = (
        pl.read_parquet(parquet_path)
        .filter(pl.col("datetime").dt.date().is_in(requested_days))
        .with_columns(pl.lit(ticker).alias("ticker"))
        .select(columns)
    )

    frames.append(ticker_df)

df = pl.concat(frames).sort(["ticker", "datetime"])
df

datetime,ticker,open,high,low,close,volume
"datetime[μs, UTC]",str,f64,f64,f64,f64,i64
2026-10-05 09:22:00 UTC,"""BEAT""",0.4399,0.4399,0.4399,0.4399,300
2026-10-05 09:43:00 UTC,"""BEAT""",0.44,0.44,0.44,0.44,217
2026-10-05 10:02:00 UTC,"""BEAT""",0.4314,0.4314,0.4314,0.4314,217
2026-10-05 11:00:00 UTC,"""BEAT""",0.4389,0.4389,0.4389,0.4389,3000
2026-10-05 11:21:00 UTC,"""BEAT""",0.4296,0.4296,0.42,0.42,10000
…,…,…,…,…,…,…
2026-10-09 23:55:00 UTC,"""VEEA""",5.04,5.05,4.99,4.99,5726
2026-10-09 23:56:00 UTC,"""VEEA""",4.98,4.98,4.95,4.95,4462
2026-10-09 23:57:00 UTC,"""VEEA""",4.98,4.98,4.9715,4.98,1456


In [29]:
GROUPS = ["ticker", "trading_day"]


def daily_vwap():
    return [
        (
            (pl.col("price") * pl.col("volume")).cum_sum().over(GROUPS)
            / pl.col("volume").cum_sum().over(GROUPS)
        ).alias("vwap_daily")
    ]


def atr(period=14):
    previous_close = pl.col("close").shift(1).over(GROUPS)

    true_range = pl.max_horizontal(
        pl.col("high") - pl.col("low"),
        (pl.col("high") - previous_close).abs(),
        (pl.col("low") - previous_close).abs(),
    )

    return [
        true_range
        .rolling_mean(window_size=period, min_samples=period)
        .over(GROUPS)
        .alias(f"atr_{period}")
    ]


def bollinger_bands(period=20, std_dev=2.0):
    middle = pl.col("close").rolling_mean(period, min_samples=period).over(GROUPS)
    deviation = (
        pl.col("close")
        .rolling_std(period, min_samples=period, ddof=0)
        .over(GROUPS)
    )

    return [
        middle.alias(f"bb_middle_{period}"),
        (middle + deviation * std_dev).alias(f"bb_upper_{period}"),
        (middle - deviation * std_dev).alias(f"bb_lower_{period}"),
    ]


def rsi(period=14):
    change = pl.col("close").diff().over(GROUPS)
    gain = pl.when(change > 0).then(change).otherwise(0.0)
    loss = pl.when(change < 0).then(-change).otherwise(0.0)

    average_gain = (
        gain.ewm_mean(alpha=1 / period, adjust=False, min_samples=period)
        .over(GROUPS)
    )
    average_loss = (
        loss.ewm_mean(alpha=1 / period, adjust=False, min_samples=period)
        .over(GROUPS)
    )

    return [
        pl.when(average_loss == 0)
        .then(100.0)
        .when(average_gain == 0)
        .then(0.0)
        .otherwise(100.0 - (100.0 / (1.0 + average_gain / average_loss)))
        .alias(f"rsi_{period}")
    ]


def donchian_channels(period=20):
    upper = pl.col("high").rolling_max(period, min_samples=period).over(GROUPS)
    lower = pl.col("low").rolling_min(period, min_samples=period).over(GROUPS)

    return [
        upper.alias(f"donchian_upper_{period}"),
        lower.alias(f"donchian_lower_{period}"),
        ((upper + lower) / 2).alias(f"donchian_middle_{period}"),
    ]


def momentum(period=10):
    return [
        (
            pl.col("close")
            - pl.col("close").shift(period).over(GROUPS)
        ).alias(f"momentum_{period}")
    ]

def mean_vol(period=100):
    mean_vol = pl.col('volume').rolling_mean(period,min_samples=period).over(GROUPS)
    return [
        mean_vol.alias(f"mean_vol_{period}")
    ]

def median_vol(period=100):
    mean_vol = pl.col('volume').rolling_median(period,min_samples=period).over(GROUPS)
    return [
        mean_vol.alias(f"median_vol_{period}")
    ]

def ema(period: int = 100):
    return [
        pl.col("close")
        .ewm_mean(span=period, adjust=False, min_samples=period)
        .over(GROUPS)
        .alias(f"ema_{period}")
    ]
def vortex(period: int = 14):
    previous_close = pl.col("close").shift(1).over(GROUPS)
    previous_high = pl.col("high").shift(1).over(GROUPS)
    previous_low = pl.col("low").shift(1).over(GROUPS)

    true_range = pl.max_horizontal(
        pl.col("high") - pl.col("low"),
        (pl.col("high") - previous_close).abs(),
        (pl.col("low") - previous_close).abs(),
    )

    positive_movement = (pl.col("high") - previous_low).abs()
    negative_movement = (pl.col("low") - previous_high).abs()

    true_range_sum = true_range.rolling_sum(period, min_samples=period).over(GROUPS)

    return [
        (
            positive_movement.rolling_sum(period, min_samples=period).over(GROUPS)
            / true_range_sum
        ).alias(f"vortex_plus_{period}"),
        (
            negative_movement.rolling_sum(period, min_samples=period).over(GROUPS)
            / true_range_sum
        ).alias(f"vortex_minus_{period}"),
    ]
def add_indicators(frame, indicator_functions):
    expressions = [
        expression
        for indicator_function in indicator_functions
        for expression in indicator_function()
    ]
    return frame.with_columns(expressions)


df_to_plot = (
    df
    .with_columns(
        pl.col("datetime").dt.convert_time_zone("America/New_York").alias("datetime")
    )
    .sort(["ticker", "datetime"])
    .with_columns(
        pl.col("datetime").dt.date().alias("trading_day"),
        ((pl.col("high") + pl.col("low") + pl.col("close")) / 3).alias("price"),
    )
)

df_to_plot = add_indicators(
    df_to_plot,
    [
        daily_vwap,
        atr,
        bollinger_bands,
        rsi,
        partial(rsi, period=3),  # RSI de 3 períodos
        donchian_channels,
        momentum,
        mean_vol,
        median_vol,
        partial(ema, 50),
        partial(ema, 10),
        vortex
    ],
)

df_to_plot

datetime,ticker,open,high,low,close,volume,trading_day,price,vwap_daily,atr_14,bb_middle_20,bb_upper_20,bb_lower_20,rsi_14,rsi_3,donchian_upper_20,donchian_lower_20,donchian_middle_20,momentum_10,mean_vol_100,median_vol_100,ema_50,ema_10,vortex_plus_14,vortex_minus_14
"datetime[μs, America/New_York]",str,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-10-05 05:22:00 EDT,"""BEAT""",0.4399,0.4399,0.4399,0.4399,300,2026-10-05,0.4399,0.4399,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2026-10-05 05:43:00 EDT,"""BEAT""",0.44,0.44,0.44,0.44,217,2026-10-05,0.44,0.439942,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2026-10-05 06:02:00 EDT,"""BEAT""",0.4314,0.4314,0.4314,0.4314,217,2026-10-05,0.4314,0.437417,null,null,null,null,null,0.769231,null,null,null,null,null,null,null,null,null,null
2026-10-05 07:00:00 EDT,"""BEAT""",0.4389,0.4389,0.4389,0.4389,3000,2026-10-05,0.4389,0.438608,null,null,null,null,null,56.820084,null,null,null,null,null,null,null,null,null,null
2026-10-05 07:21:00 EDT,"""BEAT""",0.4296,0.4296,0.42,0.42,10000,2026-10-05,0.4232,0.427389,null,null,null,null,null,18.123582,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-10-09 19:55:00 EDT,"""VEEA""",5.04,5.05,4.99,4.99,5726,2026-10-09,5.01,5.931664,0.062907,4.96998,5.046163,4.893797,54.661098,45.443224,5.09,4.9,4.995,0.0354,2312.0,1386.0,4.914584,4.985145,0.882139,0.723175
2026-10-09 19:56:00 EDT,"""VEEA""",4.98,4.98,4.95,4.95,4462,2026-10-09,4.96,5.931612,0.06005,4.96848,5.044995,4.891965,50.272795,27.133804,5.09,4.9,4.995,-0.05,2346.11,1413.5,4.915973,4.978755,0.840847,0.828952
2026-10-09 19:57:00 EDT,"""VEEA""",4.98,4.98,4.9715,4.98,1456,2026-10-09,4.977167,5.931596,0.058764,4.97048,5.045998,4.894962,53.300911,49.86056,5.09,4.9,4.995,0.01,2356.4,1440.5,4.918484,4.978981,0.798468,0.857421


In [30]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display


GROUPS = ["ticker", "trading_day"]

plot_df = (
    df_to_plot
    .sort(["ticker", "datetime"])
    .with_row_index("plot_x")
)


def add_segmented_line(fig, frame, column, name, row, color, dash="solid", width=1):
    """Añade una línea sin conectar distintos tickers ni días."""
    show_legend = True

    for segment in frame.partition_by(GROUPS, maintain_order=True):
        valid_segment = segment.drop_nulls(column)

        if valid_segment.height == 0:
            continue

        fig.add_trace(
            go.Scatter(
                x=valid_segment["plot_x"].to_list(),
                y=valid_segment[column].to_list(),
                mode="lines",
                name=name,
                legendgroup=name,
                showlegend=show_legend,
                line={"color": color, "dash": dash, "width": width},
            ),
            row=row,
            col=1,
        )

        show_legend = False


volume_colors = [
    "red" if close > open_price
    else "green" if close < open_price
    else "gray"
    for open_price, close in zip(
        plot_df["open"].to_list(),
        plot_df["close"].to_list(),
    )
]

fig = make_subplots(
    rows=5,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.025,
    row_heights=[0.50, 0.15, 0.12, 0.12, 0.11],
    subplot_titles=(
        "Precio, VWAP, Bollinger y Donchian",
        "Volumen",
        "ATR 14",
        "RSI 14",
        "Momentum 10",
    ),
)

fig.add_trace(
    go.Candlestick(
        x=plot_df["plot_x"].to_list(),
        open=plot_df["open"].to_list(),
        high=plot_df["high"].to_list(),
        low=plot_df["low"].to_list(),
        close=plot_df["close"].to_list(),
        name="Precio",
    ),
    row=1,
    col=1,
)

# Precio
add_segmented_line(fig, plot_df, "vwap_daily", "VWAP", 1,  "#1030e7", width=2)

# Bandas de Bollinger
# add_segmented_line(fig, plot_df, "bb_upper_20", "BB superior", 1, "#7f7f7f", "dot")
# add_segmented_line(fig, plot_df, "bb_middle_20", "BB media", 1, "#7f7f7f")
# add_segmented_line(fig, plot_df, "bb_lower_20", "BB inferior", 1, "#7f7f7f", "dot")

# Canales de Donchian
add_segmented_line(
    fig, plot_df, "donchian_upper_20", "Donchian superior", 1, "#9467bd", "dash"
)
# add_segmented_line(
#     fig, plot_df, "donchian_middle_20", "Donchian medio", 1, "#9467bd", "dot"
# )
add_segmented_line(
    fig, plot_df, "donchian_lower_20", "Donchian inferior", 1, "#9467bd", "dash"
)
# add_segmented_line(
#     fig, plot_df, "ema_50", "ema 50", 1, "#1dc0dd", "solid"
# )
# add_segmented_line(
#     fig, plot_df, "ema_10", "ema 10", 1, "#ca4095", "solid"
# )

# Volumen: vela alcista rojo, vela bajista verde
fig.add_trace(
    go.Bar(
        x=plot_df["plot_x"].to_list(),
        y=plot_df["volume"].to_list(),
        marker_color=volume_colors,
        name="Volumen",
    ),
    row=2,
    col=1,
)
#agrego la linea de media y mediana volumen
# add_segmented_line(fig, plot_df, "mean_vol_100", "mean_vol_100", 2, "#922909", width=2)
# add_segmented_line(fig, plot_df, "median_vol_100", "median_vol_100", 2, "#b89112", width=2)



# Indicadores inferiores
add_segmented_line(fig, plot_df, "atr_14", "ATR 14", 3, "#1f77b4", width=2)
add_segmented_line(fig, plot_df, "rsi_14", "RSI 14", 4, "#ff7f0e", width=2)
add_segmented_line(fig, plot_df, "rsi_3", "RSI 3", 5, "#2ca02c", width=2)
# add_segmented_line(fig, plot_df, "vortex_plus_14", "Vortex 14 plus", 5, "#2ca02c", width=2)
# add_segmented_line(fig, plot_df, "vortex_minus_14", "Vortex 14 minus", 5, "#96280c", width=2)

fig.add_hline(y=70, row=4, col=1, line_dash="dot", line_color="red")
fig.add_hline(y=30, row=4, col=1, line_dash="dot", line_color="green")
fig.update_yaxes(range=[0, 100], row=4, col=1)

# Una marca de eje por inicio de cada día y ticker.
day_starts = (
    plot_df
    .group_by(GROUPS, maintain_order=True)
    .agg(pl.col("plot_x").min().alias("plot_x"))
    .sort("plot_x")
)

fig.update_xaxes(
    tickmode="array",
    tickvals=day_starts["plot_x"].to_list(),
    ticktext=[
        f"{row['ticker']}<br>{row['trading_day']}"
        for row in day_starts.to_dicts()
    ],
    row=5,
    col=1,
)

# Etiqueta centrada sobre cada tramo de ticker.
ticker_ranges = (
    plot_df
    .group_by("ticker", maintain_order=True)
    .agg(
        pl.col("plot_x").min().alias("start"),
        pl.col("plot_x").max().alias("end"),
    )
)

for ticker_range in ticker_ranges.to_dicts():
    fig.add_annotation(
        x=(ticker_range["start"] + ticker_range["end"]) / 2,
        y=1.03,
        xref="x",
        yref="paper",
        text=f"<b>{ticker_range['ticker']}</b>",
        showarrow=False,
    )

fig.update_xaxes(
    showspikes=True,
    spikesnap="cursor",
    spikemode="across",
    spikecolor="#808080",
    spikethickness=1,
)

fig.update_layout(
    xaxis_rangeslider_visible=False,
    hovermode="x",
    hoversubplots="axis",
    height=1_300,
    spikedistance=-1,
)

fig.show()

Conversion de data para backtesting

In [6]:
df_to_plot

datetime,ticker,open,high,low,close,volume,trading_day,price,vwap_daily,atr_14,bb_middle_20,bb_upper_20,bb_lower_20,rsi_14,rsi_3,donchian_upper_20,donchian_lower_20,donchian_middle_20,momentum_10,mean_vol_100,median_vol_100,ema_50,ema_10,vortex_plus_14,vortex_minus_14
"datetime[μs, America/New_York]",str,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-09-24 04:00:00 EDT,"""APUS""",2.4045,2.4045,2.22,2.22,1503,2026-09-24,2.2815,2.2815,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2026-09-24 04:17:00 EDT,"""APUS""",2.38,2.38,2.38,2.38,2198,2026-09-24,2.38,2.339999,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2026-09-24 04:45:00 EDT,"""APUS""",2.25,2.25,2.25,2.25,204,2026-09-24,2.25,2.335297,null,null,null,null,null,45.070423,null,null,null,null,null,null,null,null,null,null
2026-09-24 06:58:00 EDT,"""APUS""",2.25,2.25,2.25,2.25,890,2026-09-24,2.25,2.319465,null,null,null,null,null,45.070423,null,null,null,null,null,null,null,null,null,null
2026-09-24 07:00:00 EDT,"""APUS""",2.25,2.25,2.25,2.25,316,2026-09-24,2.25,2.31517,null,null,null,null,null,45.070423,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-10-09 19:55:00 EDT,"""VEEA""",5.04,5.05,4.99,4.99,5726,2026-10-09,5.01,5.931664,0.062907,4.96998,5.046163,4.893797,54.661098,45.443224,5.09,4.9,4.995,0.0354,2312.0,1386.0,4.914584,4.985145,0.882139,0.723175
2026-10-09 19:56:00 EDT,"""VEEA""",4.98,4.98,4.95,4.95,4462,2026-10-09,4.96,5.931612,0.06005,4.96848,5.044995,4.891965,50.272795,27.133804,5.09,4.9,4.995,-0.05,2346.11,1413.5,4.915973,4.978755,0.840847,0.828952
2026-10-09 19:57:00 EDT,"""VEEA""",4.98,4.98,4.9715,4.98,1456,2026-10-09,4.977167,5.931596,0.058764,4.97048,5.045998,4.894962,53.300911,49.86056,5.09,4.9,4.995,0.01,2356.4,1440.5,4.918484,4.978981,0.798468,0.857421


In [7]:
# [c[0].upper()+c[1:].lower() for c in df_to_plot.columns]

In [8]:
data = df_to_plot.filter(pl.col('ticker')=="APUS")
data.columns = [c[0].upper()+c[1:].lower() for c in df_to_plot.columns]
data = data.to_pandas().set_index('Datetime')
data

,Ticker,Open,High,Low,Close,Volume,Trading_day,Price,Vwap_daily,Atr_14,...,Donchian_upper_20,Donchian_lower_20,Donchian_middle_20,Momentum_10,Mean_vol_100,Median_vol_100,Ema_50,Ema_10,Vortex_plus_14,Vortex_minus_14
Datetime,,,,,,,,,,,,,,,,,,,,,
2026-09-24 04:00:00-04:00,APUS,2.4045,2.4045,2.2200,2.2200,1503,2026-09-24,2.281500,2.281500,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-09-24 04:17:00-04:00,APUS,2.3800,2.3800,2.3800,2.3800,2198,2026-09-24,2.380000,2.339999,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-09-24 04:45:00-04:00,APUS,2.2500,2.2500,2.2500,2.2500,204,2026-09-24,2.250000,2.335297,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-09-24 06:58:00-04:00,APUS,2.2500,2.2500,2.2500,2.2500,890,2026-09-24,2.250000,2.319465,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-09-24 07:00:00-04:00,APUS,2.2500,2.2500,2.2500,2.2500,316,2026-09-24,2.250000,2.315170,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-24 19:55:00-04:00,APUS,5.4498,5.4498,5.4200,5.4477,3580,2026-09-24,5.439167,6.058301,0.059364,...,5.47,5.2,5.335,0.1131,3700.46,2642.0,5.261485,5.359556,1.115991,0.791361
2026-09-24 19:56:00-04:00,APUS,5.4200,5.4200,5.4100,5.4100,1494,2026-09-24,5.413333,6.058290,0.059200,...,5.47,5.2,5.335,0.0600,3709.43,2642.0,5.267309,5.368728,1.034628,0.805381
2026-09-24 19:57:00-04:00,APUS,5.4485,5.5000,5.4000,5.5000,18704,2026-09-24,5.466667,6.058164,0.061564,...,5.50,5.2,5.350,0.1303,3878.10,2784.5,5.276434,5.392595,1.018100,0.774452


In [9]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import numpy as np
from backtesting.test import SMA, GOOG


class VwapCross(Strategy):


    max_bars = 5
    valor_R=2
    def bollinger_bands_indicator(self):
        return np.vstack(
            (
                self.data.Bb_upper_20,
                self.data.Bb_middle_20,
                self.data.Bb_lower_20,
            )
        )
    def donchian_channels_indicator(self):
            return np.vstack(
                (
                    self.data.Donchian_upper_20	,
                    self.data.Donchian_lower_20,
                    self.data.Donchian_middle_20,
                )
            )
    
    def vwap_indicator(self):
        return self.data.Vwap_daily
    def atr_indicator(self):
            return self.data.Atr_14
    def momentum_indicator(self):
            return self.data.Momentum_10
    def rsi_indicator(self):
         return self.data.Rsi_14
    def init(self):
        price = self.data.Close
        self.vwap = self.I(self.vwap_indicator)
        # self.atr = self.I(self.atr_indicator)
        self.momentum= self.I(self.momentum_indicator)
        self.rsi = self.I(self.rsi_indicator)

        # self.bb_upper, self.bb_middle, self.bb_lower = self.I(
        #     self.bollinger_bands_indicator,
        #     name=("BB superior", "BB media", "BB inferior"),
        #     overlay=True,
        # )
        # self.donchian_upper, self.donchian_middle, self.donchian_lower = self.I(
        #     self.donchian_channels_indicator,
        #     name=("donchian superior", "donchian media", "donchian inferior"),
        #     overlay=True,
        # )
        self.nbars=0
        # self.ma2 = self.I(SMA, price, 20)
        self.data_ant=None

    def next(self):
        # if crossover(self.data.Close, self.data.Vwap_daily) and not self.position.is_long:
        if self.data.Close[-1]> self.data.Vwap_daily[-1] and self.data.Close[-2]< self.data.Vwap_daily[-2] and not self.trades and not self.orders:
            recent_volumes = self.data.Volume[-4:-1]
            if len(recent_volumes) == 3 and (
                self.data.Volume[-1] * 30 >= sum(recent_volumes) * 11
            ):
                # sl= self.data.Low[-3:].min()
                # tp =self.data.Close+ (self.data.Close-sl)*2
                sl= self.data.Close[-1]-self.data.Atr_14[-1]*2
                sl2= self.data.Close[-1]-self.data.Atr_14[-1]*2
                sl3= self.data.Close[-1]-self.data.Atr_14[-1]*2
                tp =self.data.Close[-1]+self.data.Atr_14[-1]*2
                tp2 =self.data.Close[-1]+self.data.Atr_14[-1]*3
                tp3 =self.data.Close[-1]+self.data.Atr_14[-1]*6
                self.nbars=0
                if sl>0 and tp>0:
                    # print(self.data.Close[-1],sl,tp)

                    size = int(self.valor_R/(self.data.Close[-1]-sl))
                    size2=int (size/4)
                    size3=int (size/4)
                    size1 = size-size2-size3
                    # print(size)
                    # print(self.data.Close[-1],sl,tp,size,size1,size2,size3)
                    # self.buy(sl=sl,tp=tp,size=size1)
                    # self.buy(sl=sl2,tp=tp2,size=size2)
                    # self.buy(sl=sl3,tp=tp3,size=size3)
                    self.buy(sl=sl, tp=tp, size=size1, tag="salida 1")
                    self.buy(sl=sl2, size=size2, tag="salida 2")
                    self.buy(sl=sl3, size=size3, tag="salida 3")

        open_price = self.data.Open[-1]
        high_price = self.data.High[-1]
        low_price = self.data.Low[-1]
        close_price = self.data.Close[-1]
        previous_open = self.data.Open[-2]
        previous_close = self.data.Close[-2]
        
        body = abs(close_price - open_price)
        upper_wick = high_price - max(open_price, close_price)
        lower_wick = min(open_price, close_price) - low_price
        upper_wick_signal = (
            body > 0
            and upper_wick > body * 3
            and lower_wick < upper_wick*1.5
        )

        bearish_engulfing_signal = (
            previous_close > previous_open      # vela anterior alcista
            and close_price < open_price        # vela actual bajista
            and open_price >= previous_close    # abre por encima del cierre previo
            and close_price <= previous_open    # cierra por debajo de la apertura previa
            and body > abs(previous_close - previous_open)
        )

        salida_1_abierta = any(
            trade.tag == "salida 1" for trade in self.trades
        )

        if not salida_1_abierta:
            for trade in self.trades:
                if trade.tag == "salida 2" and upper_wick_signal:
                    trade.close()

                if trade.tag == "salida 3" and bearish_engulfing_signal:
                    trade.close()
                # elif crossover(self.data.Close, self.data.vwap_daily):
                    # self.sell()
                # if self.position.is_long:
                #     self.nbars+=1
                #     if self.nbars>=self.max_bars:
                        
                #         self.position.close()
                self.data_ant = self.data





d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\backtesting\_plotting.py:55: UserWarning:

Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.



Loading BokehJS ...

In [10]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import numpy as np
from backtesting.test import SMA, GOOG


class VwapCross2(Strategy):


    max_bars = 5
    valor_R=2
    def bollinger_bands_indicator(self):
        return np.vstack(
            (
                self.data.Bb_upper_20,
                self.data.Bb_middle_20,
                self.data.Bb_lower_20,
            )
        )
    def donchian_channels_indicator(self):
            return np.vstack(
                (
                    self.data.Donchian_upper_20	,
                    self.data.Donchian_lower_20,
                    self.data.Donchian_middle_20,
                )
            )
    
    def vwap_indicator(self):
        return self.data.Vwap_daily
    def atr_indicator(self):
            return self.data.Atr_14
    def momentum_indicator(self):
            return self.data.Momentum_10
    def rsi_indicator(self):
         return self.data.Rsi_14
    def init(self):
        price = self.data.Close
        self.vwap = self.I(self.vwap_indicator)
        # self.atr = self.I(self.atr_indicator)
        self.momentum= self.I(self.momentum_indicator)
        self.rsi = self.I(self.rsi_indicator)

        self.nbars=0
        # self.ma2 = self.I(SMA, price, 20)
        self.data_ant=None

    def next(self):
        if self.position.is_long and self.data.Close[-1] < self.data.Vwap_daily[-1]:
            self.position.close()
            return

        # if crossover(self.data.Close, self.data.Vwap_daily) and not self.position.is_long:
        if self.data.Close[-1]> self.data.Vwap_daily[-1] and self.data.Close[-2]< self.data.Vwap_daily[-2] and not self.trades and not self.orders:
            recent_volumes = self.data.Volume[-4:-1]
            if len(recent_volumes) == 3 and (
                self.data.Volume[-1] * 30 >= sum(recent_volumes) * 11
            ) :
                # sl= self.data.Low[-3:].min()
                # tp =self.data.Close+ (self.data.Close-sl)*2
                sl= self.data.Close[-1]-self.data.Atr_14[-1]*2
                sl2= self.data.Close[-1]-self.data.Atr_14[-1]*2
                sl3= self.data.Close[-1]-self.data.Atr_14[-1]*2
                tp =self.data.Close[-1]+self.data.Atr_14[-1]*2
                tp2 =self.data.Close[-1]+self.data.Atr_14[-1]*4
                tp3 =self.data.Close[-1]+self.data.Atr_14[-1]*6
                self.nbars=0
                if sl>0 and tp>0:
                    # print(self.data.Close[-1],sl,tp)

                    size = int(self.valor_R/(self.data.Close[-1]-sl))
                    size2=int (size/4)
                    size3=int (size/4)
                    size1 = size-size2-size3
                    # print(size)
                    # print(self.data.Close[-1],sl,tp,size,size1,size2,size3)
                    self.buy(sl=sl, tp=tp,size=size1,tag="salida 1")
                    self.buy(sl=sl2, tp=tp2,size=size2,tag="salida 2")
                    self.buy(sl=sl3, tp=tp3,size=size3, tag="salida 3")
                    





In [11]:
# Experiment: retain TP1/TP2 and use a 25% runner whose stop trails the highest price after the 4-ATR threshold.
class VwapCross3(Strategy):
    max_bars = 5
    valor_R = 2

    def bollinger_bands_indicator(self):
        return np.vstack(
            (
                self.data.Bb_upper_20,
                self.data.Bb_middle_20,
                self.data.Bb_lower_20,
            )
        )

    def donchian_channels_indicator(self):
        return np.vstack(
            (
                self.data.Donchian_upper_20,
                self.data.Donchian_lower_20,
                self.data.Donchian_middle_20,
            )
        )

    def vwap_indicator(self):
        return self.data.Vwap_daily

    def atr_indicator(self):
        return self.data.Atr_14

    def momentum_indicator(self):
        return self.data.Momentum_10

    def rsi_indicator(self):
        return self.data.Rsi_14

    def init(self):
        price = self.data.Close
        self.vwap = self.I(self.vwap_indicator)
        self.momentum = self.I(self.momentum_indicator)
        self.rsi = self.I(self.rsi_indicator)
        self.nbars = 0
        self.data_ant = None
        self.runner_activation_threshold = None
        self.runner_highest_high = None

    def next(self):
        if self.position.is_long and self.data.Close[-1] < self.data.Vwap_daily[-1]:
            self.position.close()
            return

        runner_trade = next(
            (trade for trade in self.trades if trade.tag == "runner"),
            None,
        )
        if runner_trade is not None and self.runner_activation_threshold is not None:
            if self.runner_highest_high is None:
                if self.data.High[-1] >= self.runner_activation_threshold:
                    self.runner_highest_high = self.data.High[-1]
            else:
                self.runner_highest_high = max(
                    self.runner_highest_high,
                    self.data.High[-1],
                )

            if self.runner_highest_high is not None:
                trailing_stop = self.runner_highest_high - 2 * self.data.Atr_14[-1]
                if runner_trade.sl is None or trailing_stop > runner_trade.sl:
                    runner_trade.sl = trailing_stop

        if (
            self.data.Close[-1] > self.data.Vwap_daily[-1]
            and self.data.Close[-2] < self.data.Vwap_daily[-2]
            and not self.trades
            and not self.orders
        ):
            recent_volumes = self.data.Volume[-4:-1]
            if len(recent_volumes) == 3 and (
                self.data.Volume[-1] * 30 >= sum(recent_volumes) * 11
            ) and self.data.Median_vol_100 < self.data.Volume:
                atr = self.data.Atr_14[-1]
                entry_price = self.data.Close[-1]
                sl = entry_price - atr * 2
                tp1 = entry_price + atr * 2
                tp2 = entry_price + atr * 4

                self.nbars = 0
                if sl > 0 and tp1 > 0:
                    size = int(self.valor_R / (entry_price - sl))
                    size2 = int(size / 4)
                    runner_size = int(size / 4)
                    size1 = size - size2 - runner_size

                    self.runner_activation_threshold = entry_price + atr * 4
                    self.runner_highest_high = None
                    self.buy(sl=sl, tp=tp1, size=size1, tag="tp1")
                    self.buy(sl=sl, tp=tp2, size=size2, tag="tp2")
                    self.buy(sl=sl, size=runner_size, tag="runner")


In [12]:
# Experiment: retain TP1/TP2 and use a final 25% runner exited by confirmed higher-low structure break.
class VwapCross4(Strategy):
    max_bars = 5
    valor_R = 2

    def bollinger_bands_indicator(self):
        return np.vstack(
            (
                self.data.Bb_upper_20,
                self.data.Bb_middle_20,
                self.data.Bb_lower_20,
            )
        )

    def donchian_channels_indicator(self):
        return np.vstack(
            (
                self.data.Donchian_upper_20,
                self.data.Donchian_lower_20,
                self.data.Donchian_middle_20,
            )
        )

    def vwap_indicator(self):
        return self.data.Vwap_daily

    def atr_indicator(self):
        return self.data.Atr_14

    def momentum_indicator(self):
        return self.data.Momentum_10

    def rsi_indicator(self):
        return self.data.Rsi_14

    def init(self):
        price = self.data.Close
        self.vwap = self.I(self.vwap_indicator)
        # self.momentum = self.I(self.momentum_indicator)
        self.atr= self.I(self.atr_indicator)
        self.rsi = self.I(self.rsi_indicator)
        self.nbars = 0
        self.data_ant = None
        self.runner_activation_threshold = None
        self.structure_armed = False
        self.previous_pivot_low = None
        self.last_higher_low = None

    def next(self):
        if self.position.is_long and self.data.Close[-1] < self.data.Vwap_daily[-1]:
            self.position.close()
            return

        runner_trade = next(
            (trade for trade in self.trades if trade.tag == "runner"),
            None,
        )
        if runner_trade is not None and self.runner_activation_threshold is not None:
            if self.data.High[-1] >= self.runner_activation_threshold:
                self.structure_armed = True

            if self.structure_armed:
                pivot_low = self.data.Low[-2]
                is_confirmed_pivot_low = (
                    pivot_low < self.data.Low[-3]
                    and pivot_low <= self.data.Low[-1]
                )
                if is_confirmed_pivot_low:
                    if (
                        self.previous_pivot_low is not None
                        and pivot_low > self.previous_pivot_low
                    ):
                        self.last_higher_low = pivot_low
                    self.previous_pivot_low = pivot_low

                if (
                    self.last_higher_low is not None
                    and self.data.Close[-1] < self.last_higher_low
                ):
                    runner_trade.close()
                    return

        if (
            self.data.Close[-1] > self.data.Vwap_daily[-1]
            and self.data.Close[-2] < self.data.Vwap_daily[-2]
            and not self.trades
            and not self.orders
        ):
            recent_volumes = self.data.Volume[-4:-1]
            if len(recent_volumes) == 3 and (
                self.data.Volume[-1] * 30 >= sum(recent_volumes) * 11
            ) :
                atr = self.data.Atr_14[-1]
                entry_price = self.data.Close[-1]
                sl = entry_price - atr * 2
                tp1 = entry_price + atr * 2
                tp2 = entry_price + atr * 4

                self.nbars = 0
                if sl > 0 and tp1 > 0:
                    size = int(self.valor_R / (entry_price - sl))
                    size2 = int(size / 4)
                    runner_size = int(size / 4)
                    size1 = size - size2 - runner_size

                    self.runner_activation_threshold = entry_price + atr * 4
                    self.structure_armed = False
                    self.previous_pivot_low = None
                    self.last_higher_low = None
                    self.buy(sl=sl, tp=tp1, size=size1, tag="tp1")
                    self.buy(sl=sl, tp=tp2, size=size2, tag="tp2")
                    self.buy(sl=sl, size=runner_size, tag="runner")


In [13]:
bt = Backtest(data, VwapCross4, 
              commission=(0.001, 0),
            #   commission=(.001,0) 
              cash=500,
              exclusive_orders=False)
stats = bt.run(valor_R = 2)
bt.plot()

d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning:

no explicit representation of timezones available for np.datetime64



GridPlot(id='p1406', ...)

In [14]:
stats

Start                     2026-09-24 04:00...
End                       2026-09-24 19:59...
Duration                      0 days 15:59:00
Exposure Time [%]                     6.04317
Equity Final [$]                    504.04419
Equity Peak [$]                     506.00339
Commissions [$]                         0.024
Return [%]                            0.80884
Buy & Hold Return [%]               164.22069
Return (Ann.) [%]                          --
Volatility (Ann.) [%]                      --
CAGR [%]                           8194.62962
Sharpe Ratio                               --
Sortino Ratio                              --
Calmar Ratio                               --
Alpha [%]                             0.21772
Beta                                   0.0036
Max. Drawdown [%]                    -0.38719
Avg. Drawdown [%]                    -0.13994
Max. Drawdown Duration        0 days 10:55:00
Avg. Drawdown Duration        0 days 02:50:00
# Trades                          

In [15]:
stats._trades

,Size,EntryBar,ExitBar,EntryPrice,ExitPrice,SL,TP,PnL,Commission,ReturnPct,EntryTime,ExitTime,Duration,Tag,Entry_vwap_indi…,Exit_vwap_indi…,Entry_atr_indic…,Exit_atr_indic…,Entry_rsi_indic…,Exit_rsi_indic…
0,2,15,15,2.8900,3.412171,2.447829,3.412171,1.042343,0.002,0.180336,2026-09-24 08:34:00-04:00,2026-09-24 08:34:00-04:00,0 days 00:00:00,tp1,3.027088,3.027088,0.298229,0.298229,80.490732,80.490732
1,1,15,16,2.8900,3.894343,2.447829,3.894343,1.002343,0.002,0.346831,2026-09-24 08:34:00-04:00,2026-09-24 08:35:00-04:00,0 days 00:01:00,tp2,3.027088,3.524117,0.298229,0.411086,80.490732,84.036310
2,1,15,49,2.8900,6.200500,2.447829,NaN,3.308500,0.002,1.144810,2026-09-24 08:34:00-04:00,2026-09-24 09:08:00-04:00,0 days 00:34:00,runner,3.027088,5.295695,0.298229,0.569529,80.490732,63.053702
3,1,214,215,6.2000,6.160000,5.845329,NaN,-0.042000,0.002,-0.006774,2026-09-24 11:53:00-04:00,2026-09-24 11:54:00-04:00,0 days 00:01:00,runner,6.180112,6.181381,0.173100,0.186543,68.908056,73.736444
4,1,214,215,6.2000,6.160000,5.845329,6.879643,-0.042000,0.002,-0.006774,2026-09-24 11:53:00-04:00,2026-09-24 11:54:00-04:00,0 days 00:01:00,tp2,6.180112,6.181381,0.173100,0.186543,68.908056,73.736444
5,3,214,215,6.2000,6.160000,5.845329,6.534871,-0.122000,0.002,-0.006559,2026-09-24 11:53:00-04:00,2026-09-24 11:54:00-04:00,0 days 00:01:00,tp1,6.180112,6.181381,0.173100,0.186543,68.908056,73.736444
6,1,216,218,6.3522,6.170000,5.977514,NaN,-0.184200,0.002,-0.028998,2026-09-24 11:55:00-04:00,2026-09-24 11:57:00-04:00,0 days 00:02:00,runner,6.182139,6.182382,0.193829,0.205971,71.556701,66.395225
7,1,216,218,6.3522,6.170000,5.977514,7.096771,-0.184200,0.002,-0.028998,2026-09-24 11:55:00-04:00,2026-09-24 11:57:00-04:00,0 days 00:02:00,tp2,6.182139,6.182382,0.193829,0.205971,71.556701,66.395225
8,3,216,218,6.3522,6.170000,5.977514,6.723686,-0.548600,0.002,-0.028788,2026-09-24 11:55:00-04:00,2026-09-24 11:57:00-04:00,0 days 00:02:00,tp1,6.182139,6.182382,0.193829,0.205971,71.556701,66.395225
9,1,286,287,6.1600,6.130000,5.858671,NaN,-0.032000,0.002,-0.005195,2026-09-24 13:05:00-04:00,2026-09-24 13:06:00-04:00,0 days 00:01:00,runner,6.155271,6.155207,0.155157,0.149779,64.427463,59.893298


In [16]:
stats._equity_curve.to_csv("equity.csv",sep=";")

In [17]:
path_salida = os.path.join(data_dir,"output","VwapCross","results.parquet")
print(path_salida)
df = pl.read_parquet(path_salida)

E:\smallcaps_data\output\VwapCross\results.parquet


In [18]:
df.filter(pl.col("Return [%]")<0)['Return [%]'].mean()

-0.8046699447204283